# Kaggriculture Public Meta Atlas
## A living map of how public agents make decisions

**Snapshot:** 22 September 2026  
**Deep dives in this version:** `kaggriculture-17`, `V56 Harvest Ledger`, `2965+ Master Hybrid`, and `More Wheat, Smarter Sales`

> ### Disclaimer
> This notebook is a **meta-analysis of public Kaggriculture work**. It is not presented as an original competition solution.
>
> Code, strategies, experiments, names, and licenses belong to their respective authors. The goal here is to trace **decision mechanisms, idea lineage, evidence, and open directions** across public work. Original sources are linked throughout.
>
> A source can appear in this atlas at different evidence levels. A title or table of contents is enough to identify a research direction, but **not** enough to infer its implementation. Code-level claims are only made when the code or a sufficiently detailed public source was inspected.

### The question this atlas asks

A leaderboard says *who is currently scoring well*. This notebook asks something different:

> **What representations, signals, decision rules, and evaluation methods is the public community actually converging on — and where are the remaining degrees of freedom?**

The unit of analysis is therefore a **mechanism**, not a score.

# 1. Analysis framework

Every reviewed agent is decomposed into five layers:

| Layer | Question |
|---|---|
| **Policy substrate** | Is behavior generated by a fixed tape, route library, rules, search, or a learned policy? |
| **State used for adaptation** | Shops/demand? market inventory? opponent farm? opponent history? private resources? |
| **Decision locus** | Production plan, worker execution, market quantity, sale timing, order sequence, route selection? |
| **Planning horizon** | Current action, current turn, several days, phase/season? |
| **Evaluation loop** | Which opponents/seeds were used for tuning, holdout, ablation, and robustness? |

This is deliberately stricter than classifying notebooks by keywords. Two agents can both be "market-aware" while solving very different problems: one may threshold the current price; another may predict the opponent's future sale; a third may replay exact settlement order.

In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

pd.set_option('display.max_colwidth', 110)
SNAPSHOT = '2026-09-22'

# Mathematical notation used throughout the atlas

The earlier version mixed measured quantities and analyst-coded categories. This version separates them.

For agent \(i\), let \(F_i\) be the set of normalized top-level function AST hashes extracted from its submitted `main.py`.

### Pairwise implementation similarity

\[
J(i,j)=\frac{|F_i\cap F_j|}{|F_i\cup F_j|}
\]

and structural divergence

\[
D(i,j)=1-J(i,j).
\]

### Shared-core ratio

If

\[
C=\bigcap_iF_i,
\]

then

\[
R_i^{core}=\frac{|C|}{|F_i|}
\]

measures how much of agent \(i\)'s top-level implementation belongs to the common reviewed chassis.

### Branch-unique rate

If \(U_i\) is the set of normalized function bodies that appear only in agent \(i\) within the reviewed four-agent set,

\[
R_i^{unique}=\frac{|U_i|}{|F_i|}.
\]

This is **structural uniqueness in the current sample**, not a measure of historical originality.

### Evaluation notation

For \(N\) games with wins \(W\), ties \(T\), and losses \(L\):

\[
p_W=\frac{W}{N},\qquad p_T=\frac{T}{N},\qquad p_L=\frac{L}{N}.
\]

For a candidate and control evaluated on the same panel:

\[
\Delta p_W=p_W^{cand}-p_W^{ctrl}.
\]

The atlas reports this in **percentage points** because a move from 68% to 72% is a +4 pp change, not a +4% relative change.

Categorical judgments such as “defining mechanism”, “reported evidence”, or “supporting layer” are no longer plotted as if they were measured numeric magnitudes.

# 2. Public field: evidence-backed registry

This table is intentionally selective. It includes sources that reveal a distinct decision mechanism or a meaningful branch of the public meta. It is **not claimed to be an exhaustive list of every Kaggle notebook**.

Evidence levels:

- **code / detailed source** — implementation or detailed output was visible;
- **structured notebook** — title + table of contents / notebook description support the mechanism;
- **discovery signal** — useful for seeing where attention is moving, but not enough for implementation claims.

In [ ]:
PUBLIC = [
    dict(name='Reactive Agent Strategy & EDA', author='Mohammad Javad Dianat', branch='reactive economics',
         substrate='rules', adaptation='per-turn', core='irrigation priority, hiring, price buffering, crop ROI',
         evidence='detailed public discussion', url='https://www.kaggle.com/competitions/kaggriculture/discussion/739440'),
    dict(name='Kaggriculture v1', author='maverick_ss_26', branch='economic heuristic',
         substrate='rules', adaptation='per-turn', core='source-grounded economics; revenue per action',
         evidence='structured notebook', url='https://www.kaggle.com/code/maverickss26/kaggriculture-v1'),
    dict(name='Premium-First Market Agent', author='Amey Thakur', branch='market heuristic',
         substrate='rules', adaptation='per-turn', core='labor scaling, asset economics, market priority ordering',
         evidence='structured notebook', url='https://www.kaggle.com/code/ameythakur20/kaggriculture-premium-first-market-agent'),
    dict(name='Market-Smart Farming', author='tetsu2131', branch='adaptive market heuristic',
         substrate='rules / route logic', adaptation='state-aware', core='market-aware farming line; many public iterations',
         evidence='public notebook metadata', url='https://www.kaggle.com/code/tetsutani/market-smart-farming-kaggriculture'),
    dict(name='Multi-Route Farming Agent', author='Igor Zharov', branch='replay / route library',
         substrate='route / replay', adaptation='route selection', core='multi-route replay family; long public version history',
         evidence='public notebook metadata', url='https://www.kaggle.com/code/flexonafft/kaggriculture-adaptive-replay-agent'),
    dict(name='Adaptive Public-State Multi-Route', author='yamakawanin', branch='hybrid route router',
         substrate='embedded experts', adaptation='public-state router', core='outer route selection + market overlay + execution guards',
         evidence='structured notebook', url='https://www.kaggle.com/code/yamakawanin/kaggriculture-adaptive-public-state-multi-route'),
    dict(name='v48 Fast Routes', author='Kaito Fukami', branch='sparse closed-loop routing',
         substrate='route library', adaptation='observable-event routing', core='six routes, four observable events; clone/public-state branches',
         evidence='detailed notebook/log output', url='https://www.kaggle.com/code/kaitofukami/40-40-early-floor-39-46-top-10-v48-fast-routes'),
    dict(name='93.8% Public State Router', author='Thomas Tschinkel', branch='public-state routing',
         substrate='route library', adaptation='market-driven routing', core='safe six-day slices; pure market-driven routing; ablations',
         evidence='structured notebook', url='https://www.kaggle.com/code/thomastschinkel/kaggriculture-93-8-win-rate-public-state-router'),
    dict(name='Kaggriculture_reactive_router', author='Yaxon / aurax7', branch='reactive routing',
         substrate='router', adaptation='reactive', core='public reactive-router branch',
         evidence='public notebook metadata', url='https://www.kaggle.com/code/aurax7/kaggriculture-reactive-router'),
    dict(name='V23 — Adaptive Routes & Smart Sales', author='Ahmed Berat Özer', branch='heuristic lineage',
         substrate='route + wrappers', adaptation='state-aware', core='adaptive routes + sales; seed of long public lineage',
         evidence='public notebook metadata', url='https://www.kaggle.com/code/ahmedberatozer/notebook865729c24e'),
    dict(name='V34 — Observed Market Timing', author='Ahmed Berat Özer', branch='opponent-aware market',
         substrate='wrapper stack', adaptation='opponent/market', core='observed market timing + held-out/reacting-policy evaluation',
         evidence='structured notebook', url='https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v34-observed-market-timing'),
    dict(name='V38 — Smarter Feed, Stronger Margins', author='Ahmed Berat Özer', branch='economic refinement',
         substrate='wrapper stack', adaptation='opponent/market', core='feed/margin refinements layered on earlier line',
         evidence='structured notebook', url='https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v38-smarter-feed-stronger-margins'),
    dict(name='V43 — Recovering Lost Harvests', author='Ahmed Berat Özer', branch='production + market lineage',
         substrate='wrapper stack', adaptation='opponent/market', core='harvest recovery on top of a long accumulated lineage',
         evidence='structured notebook', url='https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v43-recovering-lost-harvests'),
    dict(name='Harvest Ledger V54 — Productive Idle Workers', author='haodou092', branch='worker/productivity refinement',
         substrate='heuristic lineage', adaptation='execution repair', core='productive idle-worker branch',
         evidence='public notebook metadata', url='https://www.kaggle.com/code/haodou092/notebookdb6965aa8e'),
    dict(name='V56 — Harvest Ledger — Stock-Aware Sales', author='attributed public derivative',
         branch='execution semantics / stock accounting',
         substrate='V54-style wrapper stack', adaptation='execution repair',
         core='delayed wheat harvest + remaining-stock-aware sale compaction',
         evidence='full uploaded notebook + extracted main.py', url='uploaded notebook'),
    dict(name='2965+ Master Ultra — Hybrid Engine', author='attributed public synthesis',
         branch='broad deterministic patch stack',
         substrate='Pipe/Metav4-style wrapper stack', adaptation='multi-layer reactive guards',
         core='late input trimming + overflow reclaim + 3-turn sale advance + queue closure',
         evidence='full uploaded notebook + extracted main.py', url='uploaded notebook'),
    dict(name='More Wheat, Smarter Sales', author='Dmitrii Gluzdov / attributed public integration',
         branch='integrated order-book + resource-efficiency stack',
         substrate='k17-like wrapper stack + terminal search', adaptation='market + resource guards',
         core='order-book execution + larger terminal search + fertilizer/seed budget guards',
         evidence='full uploaded notebook + embedded per-game evaluation',
         url='https://www.kaggle.com/code/dmitriigluzdov/kaggriculture-more-wheat-smarter-sales'),
    dict(name='kaggriculture-17 — Order Book', author='leoprovorov', branch='market microstructure',
         substrate='V55 wrapper stack + local search', adaptation='clone-conditioned market', core='order-slot search + tomato projection + 4 retuned constants',
         evidence='full uploaded notebook + extracted main.py', url='https://www.kaggle.com/code/leoprovorov/kaggriculture-17'),
]
public = pd.DataFrame(PUBLIC)
public

## 2.1 What is changing *right now*

The Kaggle code/discussion surface on 21–22 September adds several signals that are too fresh to classify at code level yet:

- `Demand-Preserving Turn Sale Timing`
- `Ready Stock, Earlier Sales`
- `From Orders to Actual Trades`
- `Kaggriculture V49: Funded sale timing and worker ...`
- `Kaggriculture V50 — Early Yarn Commit`
- `Game-Theoretic Master: Discrete Optimization`
- new discussion activity around RL learning, self-play, and reaching the silver zone

These titles do **not** prove how the implementations work. They do show where public attention is moving: sale execution/timing remains active, while the learned-policy branch is becoming more visible.

Discovery surfaces used for this snapshot:

- https://www.kaggle.com/code
- https://www.kaggle.com/discussions

# 3. Mechanism map — not a leaderboard

The matrix below encodes **what problem each branch is trying to solve**.

`0` = not central / absent in the inspected public description  
`1` = present  
`2` = defining mechanism  
`?` = not verified deeply enough

The purpose is to reveal *clusters of ideas*, not score agents.

In [ ]:
features = pd.DataFrame([
    ['Reactive economics',            0,0,2,1,0,0,0,0],
    ['Premium-first market',          0,0,2,1,0,0,0,0],
    ['Market-smart farming',          0,1,2,1,np.nan,0,0,0],
    ['Multi-route replay',            2,2,0,0,0,0,0,0],
    ['Adaptive public-state route',   2,2,1,1,1,0,0,0],
    ['Kaito sparse closed-loop',      2,2,1,1,2,1,0,0],
    ['Thomas market-driven router',   2,2,2,1,1,0,0,0],
    ['Ahmed V34→V43 lineage',         1,1,2,2,2,1,0,0],
    ['Aurax reactive router',         np.nan,2,1,np.nan,np.nan,0,0,0],
    ['kaggriculture-17',              1,1,2,2,2,2,2,0],
    ['RL / BC public branch',         0,2,1,1,1,0,0,2],
], columns=['branch','Replay / tape','Route routing','Market economics','Market timing',
            'Opponent model','Settlement model','Online local search','Learned policy']).set_index('branch')

symbol = {0:'—', 1:'○ supporting', 2:'● defining'}
mechanism_table = features.applymap(lambda v: '?' if pd.isna(v) else symbol[int(v)])
mechanism_table

attention=[]
for col in features.columns:
    observed=features[col].dropna()
    defining=(observed==2).sum()
    attention.append([col, defining, len(observed), defining/len(observed)])
attention=pd.DataFrame(attention,columns=['mechanism','defining_branches','observed_branches','defining_share'])
attention=attention.sort_values('defining_share')

fig, ax = plt.subplots(figsize=(9.5,5.8))
ax.barh(attention['mechanism'], attention['defining_share']*100)
ax.set_xlabel('Branches where mechanism is defining (%)')
ax.set_ylabel('')
ax.set_xlim(0,100)
ax.set_title('Where public branches concentrate their distinctive logic', pad=14, fontsize=14)
for y,(_,r) in enumerate(attention.iterrows()):
    ax.text(r['defining_share']*100+1.2,y,f"{int(r['defining_branches'])}/{int(r['observed_branches'])}",va='center',fontsize=9)
ax.grid(axis='x',alpha=0.2)
plt.tight_layout(); plt.show()

### Reading this section

The table is categorical. The bar chart answers one narrower question:

\[
A_m=\frac{\#\{\text{branches where mechanism }m\text{ is defining}\}}{\#\{\text{branches observed for }m\}}
\]

So it is a **coded attention frequency**, not a performance score. It helps identify where the public community is spending distinctive engineering effort.

# 4. Code-level similarity of the four deeply reviewed agents

The previous version used an analyst-assigned “strategy space” scatter. That was too subjective.

For the four uploaded agents we can compare the actual submitted Python.

### Method

1. decompress/extract each embedded `main.py`;
2. parse it with Python's AST;
3. normalize each top-level function by dropping source-location metadata;
4. hash the normalized function AST;
5. compute pairwise Jaccard similarity over those function hashes.

This measures **implementation overlap**, not strategic quality.


In [ ]:
CODE_SIM = pd.DataFrame(
    [
        [1.000, 0.877, 0.912, 0.972],
        [0.877, 1.000, 0.863, 0.878],
        [0.912, 0.863, 1.000, 0.886],
        [0.972, 0.878, 0.886, 1.000],
    ],
    index=['kaggriculture-17','2965+ Master','Harvest Ledger','More Wheat'],
    columns=['kaggriculture-17','2965+ Master','Harvest Ledger','More Wheat'],
)

fig, ax = plt.subplots(figsize=(8.8,7.0))
im=ax.imshow(CODE_SIM.values,vmin=0.84,vmax=1.0,aspect='equal')
ax.set_xticks(range(len(CODE_SIM.columns)))
ax.set_xticklabels(CODE_SIM.columns,rotation=28,ha='right')
ax.set_yticks(range(len(CODE_SIM.index)))
ax.set_yticklabels(CODE_SIM.index)
for r in range(CODE_SIM.shape[0]):
    for c in range(CODE_SIM.shape[1]):
        ax.text(c,r,f"{CODE_SIM.iloc[r,c]:.3f}",ha='center',va='center',fontsize=11)
ax.set_title('The deeply reviewed agents share most of their function-level implementation',pad=16,fontsize=14)
ax.set_xlabel('Compared agent')
ax.set_ylabel('Reference agent')
fig.colorbar(im,ax=ax,fraction=0.045,pad=0.04,label='Jaccard similarity J(i,j)')
plt.tight_layout(); plt.show()

pairs=[]
names=list(CODE_SIM.index)
for a in range(len(names)):
    for b in range(a+1,len(names)):
        j=float(CODE_SIM.iloc[a,b])
        pairs.append([f"{names[a]} ↔ {names[b]}",j,1-j])
pd.DataFrame(pairs,columns=['pair','Jaccard similarity','structural divergence']).sort_values('structural divergence')

### What the code similarity changes

The strongest pair is **`kaggriculture-17` ↔ `More Wheat, Smarter Sales` (0.972)**.

The relationship is tighter than the heatmap alone suggests:

- every unique function name in `kaggriculture-17` is also present in `More Wheat`;
- all shared literal module-level constants are identical;
- one inherited terminal-planner function changes its budget from **64 simulations / 4 proposals per actor** to **128 / 8**;
- `More Wheat` then appends action-changing `E410` (fertilizer guard) and `E402` (late seed budget), plus a telemetry wrapper.

Across **all four agents**, 194 normalized top-level function definitions are identical.

The correct mental model is therefore **shared public chassis + late wrapper branches**.


# 5. Idea flow in the public meta

This graph separates **policy substrate innovations** from **market-resolution innovations**. Edges mean public attribution, version lineage, or an explicitly described continuation of an earlier branch; they do not imply the entire child agent was copied from the parent.

In [ ]:
fig, ax = plt.subplots(figsize=(15,8.5))

nodes = {
    'Basic market heuristics':(0,3), 'Observed market timing':(2,3),
    'Sale reservation / race':(4,3), 'Lockstep settlement':(6,3), 'Order-slot search':(8,3),
    'Shop-router / tapes':(0,2), 'Multi-route replay':(2,2.25), 'Sparse closed-loop':(2,1.75),
    'Public-state routing':(4.5,2), 'Opponent / clone classification':(6.5,2),
    'Replay trajectories':(0,0.8), 'BC / PPO meta-policy':(2.6,0.8), 'Self-play / learned meta':(5.4,0.8),
    'Simulator / search':(2.6,0.05), 'Search-generated targets':(5.4,0.05),
}
edges=[
    ('Basic market heuristics','Observed market timing'),('Observed market timing','Sale reservation / race'),
    ('Sale reservation / race','Lockstep settlement'),('Lockstep settlement','Order-slot search'),
    ('Shop-router / tapes','Multi-route replay'),('Shop-router / tapes','Sparse closed-loop'),
    ('Multi-route replay','Public-state routing'),('Sparse closed-loop','Public-state routing'),
    ('Public-state routing','Opponent / clone classification'),('Opponent / clone classification','Order-slot search'),
    ('Replay trajectories','BC / PPO meta-policy'),('BC / PPO meta-policy','Self-play / learned meta'),
    ('Simulator / search','Search-generated targets')]

for label,(x,y) in nodes.items():
    ax.text(x,y,label,ha='center',va='center',fontsize=8.5,bbox=dict(boxstyle='round,pad=0.42'))
for a,b in edges:
    x1,y1=nodes[a]; x2,y2=nodes[b]
    ax.annotate('',xy=(x2-0.55,y2),xytext=(x1+0.55,y1),arrowprops=dict(arrowstyle='->',lw=1.3))

ax.text(-1.1,3,'MARKET',ha='right',va='center',fontsize=10)
ax.text(-1.1,2,'ROUTING',ha='right',va='center',fontsize=10)
ax.text(-1.1,0.45,'LEARNING / SEARCH',ha='right',va='center',fontsize=10)
ax.set_xlim(-2,9); ax.set_ylim(-0.6,3.6); ax.axis('off')
ax.set_title('Public idea flow separates into three interacting research lanes',pad=16,fontsize=14)
plt.tight_layout(); plt.show()

### The important shift

The market branch is increasing its **resolution**:

`what to sell → when to sell → reserve against opponent sales → model same-turn settlement → choose the exact order slots`

The route branch is increasing its **feedback**:

`single replay → several routes → route selected from shops/public state → opponent/clone-sensitive routing`

The learned branch is trying to replace hand-designed selection with learned macro decisions, but public discussion repeatedly identifies long-horizon credit assignment and coherent exploration as the hard parts.

# 6. Evaluation has become part of the meta

A major public discussion point is that evaluation itself can create false confidence.

A strong comment in the RL discussion describes two specific practices:

1. treat **one seed played in both seats** as the statistical unit;
2. keep a **seed block never inspected while tuning**;
3. separate replay games where the original shop sequence is preserved from games where RNG causes shops to re-roll.

The last issue is specific to this environment: weed spawning consumes random draws, so changing your own farm can alter future shop unlocks. A recorded opponent may then be executing a plan for a world it never saw.

Source: https://www.kaggle.com/competitions/kaggriculture/discussion/740792

This matters for every atlas entry. From now on, each deep dive should audit **evaluation topology**, not just report W/L.

# Deep Dive 001 — `kaggriculture-17`
## Your Market List Is an Order Book

### Executive interpretation

The uploaded notebook presents three modifications on top of V55:

1. an exact order-list search (`Layer D`);
2. a price/inventory-based gate for the late tomato investment;
3. four re-measured market constants.

The strongest reading of the code is more specific:

> **This is a clone/lineage-specialized market-execution optimizer attached to an extremely deep wrapper stack.**

The important contribution is the **granularity of market modelling**, while the main weakness is that the published headline evidence bundles several changes and is heavily concentrated on closely related public agents.

# 7. What is actually inside the submitted agent?

Static inspection of the decompressed `main.py` from the uploaded notebook gives:

- **1,026,965 bytes**
- **6,918 lines**
- **209 top-level functions**
- dozens of successive `agent` wrappers / parent captures
- final entry point: `_cxd_agent`

The opening attribution comments name a long chain of reused or adapted public work: `yhay81`, `prvsiyan`, `Dmitrii Gluzdov`, `aurax7`, `tetsutani`, `lucifer19`, `thomastschinkel`, and Ahmed Berat Özer's V25–V55 line.

This is therefore best analysed as a **stack of interventions**, not one coherent policy learned or designed at once.

In [ ]:
complexity = pd.DataFrame([
    ['main.py bytes', 1_026_965],
    ['source lines', 6_918],
    ['top-level functions', 209],
    ['new k17 section starts near line', 6_660],
], columns=['measure','value'])
complexity

## 7.1 Provenance graph of the reviewed source

This graph is built from attribution/change notices in the decompressed source. It is an **adaptation graph**, not a claim of exclusive authorship.

In [ ]:
fig, ax = plt.subplots(figsize=(15,8.5))

nodes = {
 'yhay81\nshop-router tapes':(0,4.8), 'prvsiyan\nproduction / tomato':(0,3.6),
 'aurax7\nstorage guard':(0,2.4), 'Dmitrii Gluzdov\nterminal / economics':(0,1.2),
 'lucifer19\nHarvest Nocturne':(0,0), 'tetsutani\npublic work':(0,-1.2),
 'V25 opening +\nproduction routes':(3,4.2),'V31+ production /\neconomic lineage':(3,3.0),
 'V28+ repair layers':(3,1.8),'V34 observed-market\nbranch':(3,0.6),'V25–V39\nintegration line':(5.4,2.3),
 'thomastschinkel\nV39 / v9':(5.4,0.4),'Ahmed V43/V44/V48\nmarket layers':(7.4,0.4),
 'V55 stack':(7.4,2.5),
 '4 retuned\nconstants':(10,3.7),'tomato\nvalue gate':(10,2.5),'order-slot\nsearch':(10,1.3)
}
edges=[
 ('yhay81\nshop-router tapes','V25 opening +\nproduction routes'),
 ('prvsiyan\nproduction / tomato','V31+ production /\neconomic lineage'),
 ('aurax7\nstorage guard','V28+ repair layers'),('Dmitrii Gluzdov\nterminal / economics','V28+ repair layers'),
 ('lucifer19\nHarvest Nocturne','V34 observed-market\nbranch'),('tetsutani\npublic work','V25–V39\nintegration line'),
 ('V25 opening +\nproduction routes','V25–V39\nintegration line'),
 ('V31+ production /\neconomic lineage','V25–V39\nintegration line'),('V28+ repair layers','V25–V39\nintegration line'),
 ('V34 observed-market\nbranch','V25–V39\nintegration line'),('thomastschinkel\nV39 / v9','V55 stack'),
 ('V25–V39\nintegration line','V55 stack'),('Ahmed V43/V44/V48\nmarket layers','V55 stack'),
 ('V55 stack','4 retuned\nconstants'),('V55 stack','tomato\nvalue gate'),('V55 stack','order-slot\nsearch'),
 ('Ahmed V43/V44/V48\nmarket layers','order-slot\nsearch')]

for label,(x,y) in nodes.items():
    ax.text(x,y,label,ha='center',va='center',fontsize=8,bbox=dict(boxstyle='round,pad=0.42'))
for a,b in edges:
    x1,y1=nodes[a]; x2,y2=nodes[b]
    ax.annotate('',xy=(x2-0.45,y2),xytext=(x1+0.45,y1),arrowprops=dict(arrowstyle='->',lw=1.1))
ax.text(0,5.5,'PUBLIC CONTRIBUTORS',ha='center',fontsize=10)
ax.text(3,5.5,'INTERMEDIATE LAYERS',ha='center',fontsize=10)
ax.text(6.4,5.5,'INTEGRATION',ha='center',fontsize=10)
ax.text(10,5.5,'k17 ADDITIONS',ha='center',fontsize=10)
ax.set_xlim(-1.4,11.2); ax.set_ylim(-2,6); ax.axis('off')
ax.set_title('kaggriculture-17 is a late wrapper on a long public provenance chain',pad=16,fontsize=14)
plt.tight_layout(); plt.show()

# 8. Where `kaggriculture-17` intervenes in the decision pipeline

The new logic is late in the pipeline. It does not redesign farm production from scratch.

**Upstream V55** produces a market list using route/tape logic, production modules, repair guards, opponent/race logic, storage constraints, and many hand-tuned constants.

Then the reviewed notebook changes the result in three places:

1. **four constants** change how aggressively existing market modules fire;
2. **tomato gate** changes one late investment decision at step/day boundary;
3. **Layer D** takes the already-produced market list and searches alternative order placements.

In [ ]:
fig, ax = plt.subplots(figsize=(13,4.5))
labels = [
    ('V55 policy stack', 0.5, 0.5),
    ('retuned\nmarket constants', 3.0, 0.5),
    ('tomato\nvalue gate', 5.6, 0.5),
    ('Layer D\norder-slot search', 8.1, 0.5),
    ('final market list', 10.8, 0.5),
]
for label,x,y in labels:
    ax.text(x,y,label,ha='center',va='center',bbox=dict(boxstyle='round,pad=0.5'))
for i in range(len(labels)-1):
    x1=labels[i][1]+0.9; x2=labels[i+1][1]-0.9
    ax.annotate('',xy=(x2,0.5),xytext=(x1,0.5),arrowprops=dict(arrowstyle='->',lw=1.5))
ax.text(0.5,0.1,'production + routes + guards + existing market logic',ha='center',fontsize=8)
ax.text(8.1,0.1,'local execution optimization; production remains inherited',ha='center',fontsize=8)
ax.set_xlim(-0.8,12); ax.set_ylim(-0.2,1.2); ax.axis('off')
ax.set_title('Where the new notebook changes the inherited policy')
plt.tight_layout(); plt.show()

# 9. Layer D — what the code really optimizes

The notebook description is broadly correct: the game settles both market lists in lockstep, so order position affects the price path.

The implementation:

- extracts SELL orders that are safe to move;
- allows them to occupy slots currently held by SELLs or fixed-price purchases (`HIRE`, `BUY_SEED`, `BUY_ANIMAL`, `BUY_LAND`);
- preserves the relative order of fixed-price operations;
- excludes `BUY_PRODUCT` interactions and wash sells from the movable set;
- scores candidates with V48's per-unit lockstep margin evaluator;
- keeps the original order as the baseline;
- accepts only a strictly better candidate;
- evaluates at most **800 candidate orderings per turn**.

That is a meaningful local best-response search. But there are two important restrictions.

### Formal objective of Layer D

Let \(\pi_0\) be the original market-list ordering and \(\hat{o}\) the predicted opponent list. Let \(M(\pi,\hat{o})\) be the money outcome returned by the exact lockstep settlement replay for candidate ordering \(\pi\).

The layer is approximately solving

\[
\pi^*=\arg\max_{\pi\in\mathcal C_B}
\left[M(\pi,\hat{o})-M(\pi_0,\hat{o})\right],
\]

where \(\mathcal C_B\) is the candidate set after the evaluation budget \(B=800\) is applied.

This makes the dependency explicit: the optimizer can be locally exact for a tested ordering while still being wrong about the opponent list \(\hat{o}\).

## 9.1 Restriction 1 — the opponent model is much narrower than the prose suggests

The source declares:

```python
_CXD_MODELS = []
```

and later uses:

```python
models = [m for m in _CXD_MODELS if m] or [orders]
```

Static inspection finds **no later assignment that populates `_CXD_MODELS`**.

So the actual fallback model is the agent's **own pre-D market list**. This is exact when the rival is effectively the same V48/V55-family policy in the same state, which is exactly the scenario highlighted by the source comments.

### Consequence

Layer D should be described as:

> **exact local best-response ordering against a clone/self-model, with strong relevance to a low-entropy public lineage**

rather than as a general opponent-model optimizer.

This also makes the 40–0 V55 mirror result easier to interpret: the mechanism is architecturally aligned with that matchup.

## 9.2 Restriction 2 — “exact” is conditional on an 800-evaluation budget

If there are `n` mutable slots and `k` movable SELL orders, the generator can enumerate roughly:

\[
P(n,k)=\frac{n!}{(n-k)!}
\]

placements before considering the 800-candidate cap.

In [ ]:
rows=[]
BUDGET=800
for k in [2,3,4,5]:
    for s in range(k,11):
        omega=math.perm(s,k)
        rows.append((s,k,omega,min(1.0,BUDGET/omega)))
coverage=pd.DataFrame(rows,columns=['free_slots','sell_orders','orderings','coverage'])

fig, ax = plt.subplots(figsize=(9.8,5.9))
for k,g in coverage.groupby('sell_orders'):
    ax.plot(g.free_slots,g.coverage*100,marker='o',label=f'{k} SELL orders')
ax.axhline(100,linewidth=1)
ax.set_xlabel('Free market slots s')
ax.set_ylabel('Permutation space evaluated (%)')
ax.set_ylim(0,105)
ax.set_title('The 800-candidate budget becomes partial as the market list grows',pad=14,fontsize=14)
ax.legend(title='Orders to place')
ax.grid(alpha=0.2)
ex=coverage[(coverage.free_slots==7)&(coverage.sell_orders==4)].iloc[0]
ax.annotate(f"P(7,4)={int(ex.orderings)}\ncoverage={ex.coverage*100:.1f}%",xy=(7,ex.coverage*100),xytext=(7.35,72),arrowprops=dict(arrowstyle='->'))
plt.tight_layout(); plt.show()

The number of placements of \(k\) distinguishable SELL orders into \(s\) free slots is

\[
\Omega(s,k)=P(s,k)=\frac{s!}{(s-k)!}.
\]

With budget \(B=800\), search coverage is

\[
q(s,k)=\min\left(1,\frac{B}{\Omega(s,k)}\right).
\]

For example,

\[
P(7,4)=840,\qquad q(7,4)=800/840\approx95.2\%.
\]

Therefore Layer D uses an **exact settlement evaluator** inside a search that is exhaustive only while \(\Omega(s,k)\le800\).

# 10. Tomato gate — stronger causal variable, weak uncertainty model

The upstream trigger uses a shop-count proxy. The new gate projects the quantity that actually drives revenue:

`future tomato inventory → price curve → expected revenue`

This is a conceptually strong change. It replaces a coarse observable (`number of tomato-consuming shops`) with a closer causal variable (`inventory at the time our crop arrives`).

The code includes:

- current shared tomato inventory;
- currently visible rival tomato tiles and planting dates;
- expected future shop unlock demand;
- own planned 20 units/day on days 26–29;
- an empirical rival-tomato rate (`0.75 units/day/tile`);
- an empirical town-demand slack (`2.4 units/day`);
- a fixed revenue gate of `9,000` coins.

### What is still approximate

The projection treats several uncertain quantities as point estimates. It also carries rival tomato production to season end with a fixed empirical rate because the calibration examples replanted.

The source comments say the two empirical corrections came from replay measurements. That makes the mechanism testable, but it also creates an evaluation question: **were those replay worlds preserving the original shop RNG sequence?** Public discussion warns that this is not automatic in Kaggriculture.

### Tomato gate as a forward state model

The source narrative can be represented as

\[
I_{t+1}=I_t+S_t^{ours}+S_t^{opp}-D_t,
\]

where \(I_t\) is market inventory, the \(S\) terms are projected supply, and \(D_t\) is projected town demand.

The game engine maps inventory to price via its own price curve

\[
p_t=f(I_t),
\]

and the late investment is accepted when projected value clears the configured threshold:

\[
V=\sum_t q_t\,p_t\ge 9000.
\]

The notebook prose does not publish a standalone closed-form expression for \(f\), so the atlas leaves it as the engine function rather than inventing one. The important modeling change is replacing a shop-count proxy with a projected inventory → price → value chain.

# 11. The headline experiment — impressive, but bundled

The notebook reports this paired comparison on seven public agents:

- plain V55: **191 wins, 36 ties, 53 losses** over 280 games;
- candidate: **280 wins, 0 ties, 0 losses**.

It also reports 108 recorded ladder games:

- plain V55: **74–34**;
- candidate: **78–30**.

These two evaluation surfaces tell very different stories.

In [ ]:
local_base = 191/280*100
local_new = 280/280*100
ladder_base = 74/108*100
ladder_new = 78/108*100
compare = pd.DataFrame([
    ['Curated 7-agent local panel', local_base, local_new, local_new-local_base],
    ['Recorded ladder games', ladder_base, ladder_new, ladder_new-ladder_base],
], columns=['evaluation_surface','V55 win share','candidate win share','change_pp'])
compare

In [ ]:
fig, ax = plt.subplots(figsize=(9.4,5.8))
x=np.arange(len(compare)); w=0.34
ax.bar(x-w/2,compare['V55 win share'],width=w,label='plain V55')
ax.bar(x+w/2,compare['candidate win share'],width=w,label='candidate')
ax.set_xticks(x); ax.set_xticklabels(['Curated public panel','Recorded ladder'])
ax.set_ylabel('Wins / all games (%)')
ax.set_ylim(0,105)
ax.set_title('The measured uplift shrinks sharply outside the curated local panel',pad=14,fontsize=14)
ax.legend(); ax.grid(axis='y',alpha=0.2)
for i,r in compare.iterrows():
    ax.text(i,max(r['V55 win share'],r['candidate win share'])+2,f"Δ {r['change_pp']:+.1f} pp",ha='center',fontsize=10)
plt.tight_layout(); plt.show()

Numerically,

\[
\Delta p_W^{local}=\frac{280}{280}-\frac{191}{280}=+31.8\text{ pp},
\]

while

\[
\Delta p_W^{ladder}=\frac{78}{108}-\frac{74}{108}=+3.7\text{ pp}.
\]

The ratio of the observed percentage-point uplifts is roughly

\[
\frac{31.8}{3.7}\approx8.6.
\]

That is a **generalization warning signal**, not proof of overfitting: the two panels have different composition and dependence structure.

### Why this is the central evidence issue

The local panel moves by **+31.8 percentage points of win share**. The recorded ladder sample moves by about **+3.7 points**.

That does not make the local experiment invalid. It means it measures something narrower:

- the seven agents are contemporaneous public solutions;
- the notebook itself says many of them share substantial code;
- Layer D is especially well matched to clone/same-lineage market behavior;
- four constants were re-measured against this same current field;
- the tomato threshold was also selected after comparison on live/mirror evidence.

So the 280-game sweep is best treated as **strong evidence of a matchup-family exploit**, not yet as proof of equally large out-of-family generalization.

# 12. Outcome decomposition

The local change is not merely “fewer losses.” It converts every tie and loss in the reported panel into a win.

In [ ]:
outcomes = pd.DataFrame({
    'policy':['plain V55','kaggriculture-17'],
    'wins':[191,280],
    'ties':[36,0],
    'losses':[53,0],
}).set_index('policy')

fig, ax = plt.subplots(figsize=(8,5))
bottom=np.zeros(len(outcomes))
for col in ['wins','ties','losses']:
    ax.bar(outcomes.index,outcomes[col],bottom=bottom,label=col)
    bottom += outcomes[col].values
ax.set_ylabel('Games (280 total per policy)')
ax.set_title('Reported local panel: all non-wins disappear in the bundled candidate')
ax.legend()
plt.tight_layout(); plt.show()

# 13. Causal attribution problem: three changes are evaluated together

The strongest limitation of the published analysis is **not sample size**. It is attribution.

The final candidate contains:

- Layer D order search;
- tomato value gate;
- four re-measured constants.

The 280-game headline comparison evaluates the **bundle**.

The notebook provides useful side evidence:

- individual constants reportedly turn 12–24 wins on their own over 280 games;
- forcing the tomato investment regardless reportedly costs 34 wins in 119 games;
- the author says the four constants have non-identical effects.

But the public summary does not provide the clean staged experiment needed to answer:

> **How much of 191–53 → 280–0 is Layer D itself?**

A minimal causal ablation would be:

`V55 → +D → +D+tomato → +D+tomato+constants`

using the same frozen holdout seed-pairs, followed by an out-of-lineage opponent panel.

# 14. Parameter search: winner's-curse risk

The source says V55 has 69 constants, and 26 were swept at two candidate values each, with 50 games per candidate against the same opponent/seed environment. Seven settings were selected as improvements; four market-side constants were kept here.

This is a legitimate engineering search, but it creates a classic **selection bias**:

- many variants are observed;
- the winners are selected from noisy estimates;
- the headline evaluation remains close to the selection distribution.

For this kind of competition, a better atlas standard is:

1. tune on opponent-family × seed-pair block A;
2. freeze all knobs;
3. report block B once;
4. then test on a **different lineage family**, not only new versions of the same public stack.

The public discussion on seed/seat pairing and untouched holdout blocks makes this particularly important here.

# 15. Source inconsistency audit

The uploaded notebook contains a numerical inconsistency worth preserving in the record.

The per-opponent table reports margin gains of roughly **+735 to +824 per game**, and the total cell reports **+758**.

The following markdown sentence says the gain is **+124 to +133 per game**.

Those cannot both describe the same measurement. The structured table and total agree with each other, so this atlas treats `+124…+133` as a likely stale narrative line.

This is a small example of why every future deep dive will separate:

- **code-derived facts**;
- **structured experiment output**;
- **author interpretation**;
- **our interpretation**.

# 16. What `kaggriculture-17` actually adds to the community map

The useful conceptual contribution is not “another high-scoring V55 variant.” It is a **new decision granularity**.

Earlier market modules in this lineage increasingly model:

1. current economic value;
2. observed opponent timing;
3. future sale reservation / race;
4. same-turn lockstep settlement;
5. **the exact slot position of each sale inside the market list**.

That last step turns the market list itself into a small combinatorial optimization problem.

### What it does *not* yet solve

It does not provide a general multi-opponent market belief model. It does not search multiple future turns. It does not learn when its self/clone opponent assumption is wrong. And it does not isolate Layer D's causal effect from the other published changes.

Those are precisely the next research directions suggested by the analysis.

# 17. Where the community appears to be looking now

## Branch A — replay → routing → closed loop

Public work moved beyond pure static tapes. The important question is now how to select among strong continuations using shops, market state, and sparse opponent cues.

Evidence:
- Igor Zharov's multi-route replay family;
- yamakawanin's embedded experts + outer selector;
- Kaito's six routes / four observable events;
- Thomas Tschinkel's market-driven public-state router;
- Yusuke Hayashi's public discussion of branching replay early.

## Branch B — market heuristics → market microstructure

The public Ahmed lineage, plus very recent notebook titles, shows continued concentration on sale timing, funded sales, actual trades, and order effects. `kaggriculture-17` is a natural endpoint of this branch: model the settlement at order-slot resolution.

## Branch C — learned macro policy / self-play

Earlier discussion reported atomic BC error accumulation and PPO instability. By 21–22 September, Kaggle's discussion surface shows active threads titled **“RL finally starts learning something!”**, **“RL Progress update: self play”**, and **“Finally reached the silver medal zone with Reinforcement Learning.”** Another participant publicly confirms their patterned agent is RL-based while describing the behavior as a hackable local optimum.

This suggests the public conversation is shifting from *“can RL work?”* toward *“how do we make learned policies robust and strategically diverse?”*

## Branch D — evaluation and simulator-generated supervision

A sophisticated evaluation thread argues for paired-seat seed blocks, untouched holdouts, preserved shop-sequence checks, and search-generated value targets. This may become as important as the policy architecture itself.

# 18. Strong next experiments suggested by this deep dive

These are experiments that would genuinely move the map, rather than merely retune the same agent.

| Experiment | What it tests | Why it matters |
|---|---|---|
| **Pure Layer-D ablation** | V55 vs V55+D only | isolates the actual order-search contribution |
| **Cross-lineage panel** | replay routers, reactive agents, RL-like agents, V55 family separately | tests whether clone specialization generalizes |
| **Opponent-model ensemble** | populate `_CXD_MODELS` with several inferred order policies | turns self-model best response into robust execution |
| **Belief over opponent family** | infer route/market family from first N turns | uses uncertainty instead of one assumed rival list |
| **Two-turn market search** | include next-turn inventory/liquidity consequences | tests whether local slot gain survives downstream effects |
| **Budget-aware search** | branch-and-bound / beam / heuristic ordering beyond 800 permutations | removes deterministic truncation bias |
| **Frozen holdout protocol** | paired seats + untouched seeds + shop-sequence audit | protects improvements from tuning/evaluation leakage |
| **Simulator-generated macro targets** | compare complete market plans from real states | connects heuristic/search branch to learned policy branch |

# 19. Updated atlas thesis

The public Kaggriculture meta is better described as **progressive compression of a huge decision problem into smaller controllable subproblems**.

- Replay agents compress 720 turns into a small route library.
- Routers compress adaptation into a choice among expert continuations.
- Wrapper stacks compress improvement into local failure repairs.
- Market-race agents compress opponent interaction into a few timing/reservation decisions.
- `kaggriculture-17` compresses one part of the game even further: **given an already chosen market action set, optimize only its execution order**.
- RL/search work tries to learn or generate the higher-level selector that engineered agents currently hand-design.

This is a more useful picture of the community than “heuristics vs RL.” The real contest is over **which decisions should remain scripted, which should branch, which deserve exact simulation, and which are worth learning**.

# 20. Four-agent structural comparison
## Shared chassis, different late interventions

Static inspection of the decompressed submissions gives:

| Agent | `main.py` bytes | lines | top-level functions | distinctive late direction |
|---|---:|---:|---:|---|
| `kaggriculture-17` | 1,026,965 | 6,918 | 209 | exact market-list search + tomato value gate |
| `2965+ Master` | 1,040,667 | 7,216 | 219 | broad deterministic patch stack |
| `Harvest Ledger` | 1,024,063 | ~6.7k | 202 | stock-correct sale compaction + delayed wheat |
| `More Wheat` | 1,032,755 | 7,025 | 213 | k17-like market stack + larger terminal search + input guards |

Most divergence sits close to the end of a long accumulated wrapper chain.

In [ ]:
agent_structure=pd.DataFrame([
 ['kaggriculture-17',209,194,0],['2965+ Master',219,194,16],['Harvest Ledger',202,194,6],['More Wheat',213,194,2]],
 columns=['agent','function_count','shared_core','branch_unique'])
agent_structure['core_ratio']=agent_structure.shared_core/agent_structure.function_count
agent_structure['unique_rate']=agent_structure.branch_unique/agent_structure.function_count

fig, ax = plt.subplots(figsize=(9.5,5.8))
x=np.arange(len(agent_structure))
ax.bar(x,agent_structure.core_ratio*100)
ax.set_xticks(x); ax.set_xticklabels(agent_structure.agent,rotation=18,ha='right')
ax.set_ylim(0,100); ax.set_ylabel('Shared-core ratio (%)')
ax.set_title('Most top-level function structure belongs to the common four-agent core',pad=14,fontsize=14)
for i,r in agent_structure.iterrows():
    ax.text(i,r.core_ratio*100+1.5,f"{r.core_ratio*100:.1f}%",ha='center')
    ax.text(i,r.core_ratio*100-8,f"194/{int(r.function_count)}",ha='center',fontsize=9)
ax.grid(axis='y',alpha=0.2)
plt.tight_layout(); plt.show()

In [ ]:
nov=agent_structure.sort_values('unique_rate')
fig, ax = plt.subplots(figsize=(9.5,5.5))
ax.barh(nov.agent,nov.unique_rate*100)
ax.set_xlabel('Branch-unique function bodies / all top-level function bodies (%)')
ax.set_ylabel('')
ax.set_title('Agent-specific code is a thin late layer on top of the shared chassis',pad=14,fontsize=14)
for y,(_,r) in enumerate(nov.iterrows()):
    ax.text(r.unique_rate*100+0.12,y,f"{r.unique_rate*100:.2f}% ({int(r.branch_unique)}/{int(r.function_count)})",va='center',fontsize=9)
ax.grid(axis='x',alpha=0.2)
plt.tight_layout(); plt.show()

The two ratios answer different questions:

\[
R_i^{core}=|C|/|F_i|,
\qquad
R_i^{unique}=|U_i|/|F_i|.
\]

A low unique rate does **not** mean the agent contributed no ideas historically. `kaggriculture-17`, for example, has zero branch-unique normalized function bodies in this four-agent *current* sample because later submissions inherited those bodies.

The “branch-unique” count is conservative: it counts normalized top-level function bodies found in only one of these four files. Changed constants and changed call arguments are not counted as new functions.

The engineering pattern is increasingly:

> **reuse a mature policy → identify one failure mode → append a guard/search/repair → re-test.**

## 20.1 Reviewed-code branch graph

In [ ]:
fig, ax = plt.subplots(figsize=(12.5,6.6))
nodes={
 'Shared public chassis\n194 identical function ASTs':(0,2.4),
 'Harvest Ledger\nstock semantics':(3.2,3.8),
 'kaggriculture-17\nmarket microstructure':(3.2,2.4),
 '2965+ Master\nbroad deterministic repairs':(3.2,1.0),
 'More Wheat\nintegrated branch':(6.8,2.4),
 'V56 input controls\nE402 + E410':(6.8,0.8)}
edges=[
 ('Shared public chassis\n194 identical function ASTs','Harvest Ledger\nstock semantics'),
 ('Shared public chassis\n194 identical function ASTs','kaggriculture-17\nmarket microstructure'),
 ('Shared public chassis\n194 identical function ASTs','2965+ Master\nbroad deterministic repairs'),
 ('kaggriculture-17\nmarket microstructure','More Wheat\nintegrated branch'),
 ('V56 input controls\nE402 + E410','2965+ Master\nbroad deterministic repairs'),
 ('V56 input controls\nE402 + E410','More Wheat\nintegrated branch')]
for label,(x,y) in nodes.items():
    ax.text(x,y,label,ha='center',va='center',fontsize=9,bbox=dict(boxstyle='round,pad=0.5'))
for a,b in edges:
    x1,y1=nodes[a];x2,y2=nodes[b]
    ax.annotate('',xy=(x2-0.7,y2),xytext=(x1+0.7,y1),arrowprops=dict(arrowstyle='->',lw=1.4))
ax.text(0,4.7,'COMMON CORE',ha='center',fontsize=10)
ax.text(3.2,4.7,'BRANCH-SPECIFIC INTERVENTIONS',ha='center',fontsize=10)
ax.text(6.8,4.7,'CONVERGENCE / REUSE',ha='center',fontsize=10)
ax.set_xlim(-1.5,8.2);ax.set_ylim(0,5.1);ax.axis('off')
ax.set_title('Reviewed-code lineage: a shared chassis with thin late branches',pad=14,fontsize=14)
plt.tight_layout();plt.show()

`More Wheat` is the clearest convergence point: its inspected source contains the k17 order-book/tomato machinery, then adds V56-style fertilizer/seed controls and widens the inherited terminal search.

`2965+ Master` takes a sibling path. It does not add the exact k17 order-slot search; it spends complexity on deterministic sale preemption, overflow handling, queue repair and late-input trimming.

`Harvest Ledger` is the semantics-oriented branch.

# Deep Dive 002 — V56 Harvest Ledger
## Stock-aware sale compaction

### Executive interpretation

This notebook is primarily a **correctness intervention inside the market-list postprocessor**.

Its core failure mode is subtle:

> a later SELL-compaction pass must not treat stock already consumed by an earlier SELL as still available.

That matters because market orders are positional: “cleaning up” a list can change the simultaneous interaction with the opponent.

## 21. What the code changes

The distinctive `EXP334/EXP335` path keeps a mutable `remaining` copy of the shed.

It:

1. caps each sale by currently remaining stock;
2. decrements that stock before later orders are considered;
3. converts fully ineffective sales to **empty slots**;
4. preserves BUY/HIRE positions instead of pulling them left;
5. stops aggressive mixed compaction after a relevant `BUY_PRODUCT`.

Two invariants are protected:

- **inventory conservation** — one unit cannot be sold twice;
- **market-slot semantics** — removing a no-op order must not silently reindex a purchase.

This is closer to a transaction-system correctness fix than a new farming policy.

### Stock accounting as a recurrence

For product \(p\), let \(r_{p,j}\) be remaining shed stock immediately before market-list order \(j\). If order \(j\) asks to sell \(q_j\), the executable quantity is

\[
q_j^{exec}=\min(q_j,r_{p,j}),
\]

and the remaining stock becomes

\[
r_{p,j+1}=r_{p,j}-q_j^{exec}\ge0.
\]

The bug class being prevented is equivalent to evaluating multiple SELL orders against the same initial \(r_{p,0}\), which can double-count stock. The second invariant is positional: if \(q_j^{exec}=0\), the slot may become empty, but unrelated BUY/HIRE orders are not automatically shifted into a different settlement index.

## 21.1 Delayed temporary wheat

The file also rewrites the early HybridOpening tape. The temporary wheat harvest is postponed from the earlier step-53…57 closure into a later step-84…91 sequence, adding another water/growth refresh before harvest, restoring the pasture afterwards, returning the wheat, and extending the sale quantity.

So the distinctive branch mixes **early production timing** with **stock-correct market compaction**.

## 21.2 Evidence audit

The notebook explicitly says corrected accounting is **not automatically a score improvement**.

The uploaded artifact contains hash verification and deterministic packaging, but no competitive match panel, no holdout, and no ablation table.

For the atlas this is a useful distinction: a convincing semantic fix and a demonstrated competitive gain are separate claims.

# Deep Dive 003 — 2965+ Master Hybrid Engine
## Broad deterministic repair stack

### Executive interpretation

This branch attacks many small leakages at once rather than searching one decision exactly.

Its late stack includes:

- `E402`: cap late wheat/carrot seed purchases by remaining planting demand;
- `E410`: skip fertilizer that cannot improve the planned harvest;
- `R148`: reclaim stock that would otherwise overflow at dawn;
- `ADV`: sell selected cash products up to three turns before the tape would sell them;
- queue closure: remove impossible cash-sale no-ops and pull later sales into holes;
- day-27 seed-float trim;
- day-29 fertilizer-buy knockout.

Code for next-turn seed prefunding exists, but `_R148_SEEDS=False` in the inspected configuration, so that path is inactive.

## 22. What kind of reasoning is this?

These layers mostly make **bounded deterministic forecasts of the agent's own existing plan**.

- `ADV` looks three turns ahead.
- `E402` scans the remaining route to bound seed demand.
- `R148` models the immediate end-of-day warehouse consequence.
- the final seed-float layer counts still-harvestable planting demand.

The opponent enters mainly through a mirror/front-running idea: if a similar policy will sell soon, move already-available stock earlier.

So this branch is **broader but shallower** than k17's exact local market-settlement search.

### A bounded-horizon interpretation

The `ADV` sale-preemption layer is a short-horizon rule. Schematically, for current turn \(t\), it searches the inherited tape for a planned sale of product \(p\) within

\[
1\le\Delta\le3.
\]

If stock is already available now, a quantity can be advanced from \(t+\Delta\) to \(t\), subject to the wrapper's product and safety checks.

So the distinctive planning horizon is deliberately small:

\[
H_{ADV}=3\text{ turns}.
\]

The seed/fertilizer guards use a different idea: cap purchases by **remaining realizable demand** in the inherited route. A schematic seed cap is

\[
q_{buy}\le \max(0,D_{remaining}-S_{available}),
\]

where the exact code determines \(D_{remaining}\) by scanning the future tape. This is not global planning; it is deterministic closure of known future obligations.

## 22.1 Breadth vs local depth

| Dimension | `kaggriculture-17` | 2965+ Master |
|---|---|---|
| exact order-slot search | yes | no |
| tomato inventory projection | yes | no k17 tomato layer |
| 3-turn sale preemption | no | yes |
| dawn overflow reclaim | no Master R148 layer | yes |
| late seed cap | no final E402 wrapper | yes |
| fertilizer usefulness guard | no final E410 wrapper | yes |
| day-27 seed float trim | no | yes |
| day-29 fertilizer knockout | no | yes |
| style | deep local market model | wide deterministic repair stack |

This is a genuine fork over a shared chassis.

## 22.2 Evidence audit

The markdown reports strong head-to-head numbers against several public agents and says seeds/seats were balanced.

Those pairwise results are **not reconstructed by executable competitive-evaluation cells in the uploaded notebook**. The executable cells verify the file, perform an integration/self-match style run, and package the submission.

The atlas therefore stores the head-to-head table as **author-reported evidence**, not notebook-reproduced evidence.

# Deep Dive 004 — More Wheat, Smarter Sales
## Integration plus stricter model selection

### Executive interpretation

At code level this is extremely close to `kaggriculture-17`, then adds resource-efficiency controls and increases the terminal planner budget.

The clean diff is:

1. retain the k17 order-book/tomato stack and the same four retuned constants;
2. widen terminal rescue from **64 simulations / 4 proposals per actor** to **128 / 8**;
3. append `E410` fertilizer-usefulness checking;
4. append `E402` late wheat/carrot seed-budget capping;
5. finish with a telemetry-only wrapper.

### Terminal-search budget change

The inherited terminal planner changes from

\[
B_{sim}:64\rightarrow128,
\qquad
K_{proposal}:4\rightarrow8\text{ per actor}.
\]

So both the simulation budget and per-actor proposal width are doubled. This is a clean computational change: the agent is not given a new terminal objective; it is given **more search capacity around the inherited objective**.

## 23. Why this entry matters even though a challenger was rejected

The notebook publishes a **negative selection result**.

It separates:

- a small public pilot;
- development worlds;
- an eight-world untouched confirmation block;
- both seats for every confirmation world.

The leading challenger failed the frozen gates: reported weighted outcome change **−0.0885** and mean relative-margin change **−110.4**. The previously qualified control was retained.

That tells us something about the search process, not just the final agent.

## 23.1 Embedded confirmation panel

The notebook embeds per-game records and summarizes 64 confirmation games:

| Opponent cohort | Games | W-L-T | Mean margin | Worst margin |
|---|---:|---:|---:|---:|
| Clone Race Horizon | 16 | 16-0-0 | +261.6 | +21 |
| Master Engine | 16 | 16-0-0 | +768.6 | +305 |
| Order Book v2 | 16 | 16-0-0 | +331.9 | +160 |
| Previous notebook (v7) | 16 | 16-0-0 | +157.2 | +125 |

The source itself warns that the pool shares broad ShopRouter/Metav4 ancestry. This is strong **within-lineage confirmation**, not universal robustness.

In [ ]:
mw_eval=pd.DataFrame([
 ['Clone Race Horizon',261.625,21],['Master Engine',768.625,305],['Order Book v2',331.875,160],['Previous notebook (v7)',157.25,125]],
 columns=['opponent cohort','mean margin','worst margin']).sort_values('mean margin')
fig, ax = plt.subplots(figsize=(9.6,5.8))
y=np.arange(len(mw_eval))
ax.hlines(y,mw_eval['worst margin'],mw_eval['mean margin'],linewidth=2)
ax.scatter(mw_eval['worst margin'],y,s=55,label='Worst margin')
ax.scatter(mw_eval['mean margin'],y,s=75,label='Mean margin')
ax.set_yticks(y);ax.set_yticklabels(mw_eval['opponent cohort'])
ax.set_xlabel('Money margin');ax.set_ylabel('')
ax.set_title('More Wheat confirmation: the whole observed range remained positive',pad=14,fontsize=14)
ax.legend();ax.grid(axis='x',alpha=0.2)
for yy,(_,r) in enumerate(mw_eval.iterrows()):
    ax.text(r['mean margin']+12,yy,f"mean {r['mean margin']:.0f}",va='center',fontsize=9)
plt.tight_layout();plt.show()

The confirmation chart is intentionally not merged with headline numbers from other notebooks. Different opponent panels and tuning histories make a cross-notebook “performance ranking” misleading.

# 24. Intervention-surface map

`2` = defining/distinctive late mechanism  
`1` = present but inherited/supporting  
`0` = absent from the inspected final branch

In [ ]:
surface = pd.DataFrame([
    ['Harvest Ledger', 2,2,0,0,0,0,0,0,1],
    ['kaggriculture-17', 1,0,0,2,2,1,0,0,1],
    ['2965+ Master', 1,0,2,2,0,0,2,2,1],
    ['More Wheat', 1,0,2,2,2,1,0,0,2],
], columns=['agent','opening / production timing','stock-correct compaction','input efficiency','sale timing','order-slot microstructure','investment projection','overflow / warehouse','late deterministic trimming','terminal search']).set_index('agent')

labels=surface.applymap(lambda v:'● defining' if v==2 else ('○ supporting' if v==1 else '—'))
labels

breadth=(surface==2).sum(axis=1).sort_values()
fig, ax = plt.subplots(figsize=(9.2,5.3))
ax.barh(breadth.index,breadth.values)
ax.set_xlabel('Distinct intervention surfaces coded as defining')
ax.set_ylabel('')
ax.set_title('Master and More Wheat spread new logic across more decision surfaces',pad=14,fontsize=14)
for y,v in enumerate(breadth.values): ax.text(v+0.08,y,str(int(v)),va='center')
ax.grid(axis='x',alpha=0.2)
plt.tight_layout();plt.show()

The bar chart uses the transparent breadth count

\[
B_i=\sum_m\mathbf 1[x_{i,m}=\text{defining}],
\]

where the categories themselves are analyst-coded. It should be read as **intervention breadth**, not performance or sophistication.

- **Harvest Ledger** concentrates novelty in semantic correctness and opening timing.
- **kaggriculture-17** concentrates it in market microstructure and investment valuation.
- **Master** spreads smaller deterministic repairs across more surfaces.
- **More Wheat** combines the k17 market branch with resource-efficiency and a wider terminal planner.

# 25. Evidence topology

Mechanism sophistication and evidence quality are independent.

Scale:

- `0` — absent in the uploaded notebook
- `1` — reported/summarized, underlying artifact not present
- `2` — executable or embedded data supports the claim

In [ ]:
evidence=pd.DataFrame([
 ['Harvest Ledger',2,0,0,0,0,0,0],['kaggriculture-17',2,1,2,2,0,1,1],['2965+ Master',2,2,1,1,0,0,0],['More Wheat',2,2,2,2,2,2,0]],
 columns=['agent','source integrity','full-game integration','opponent panel','paired seats','untouched holdout','selection / ablation evidence','external ladder evidence']).set_index('agent')

emap={0:'Absent',1:'Reported',2:'Verified'}
evidence_labels=evidence.applymap(lambda v:emap[int(v)])
evidence_labels

verified=(evidence==2).sum(axis=1).sort_values()
fig, ax = plt.subplots(figsize=(9.2,5.3))
ax.barh(verified.index,verified.values)
ax.set_xlim(0,evidence.shape[1])
ax.set_xlabel(f'Verified evidence dimensions (out of {evidence.shape[1]})')
ax.set_ylabel('')
ax.set_title('The uploaded notebooks differ sharply in directly inspectable evaluation coverage',pad=14,fontsize=14)
for y,v in enumerate(verified.values): ax.text(v+0.08,y,f"{int(v)}/{evidence.shape[1]}",va='center')
ax.grid(axis='x',alpha=0.2)
plt.tight_layout();plt.show()

### Why this matters

- `Harvest Ledger` is mainly a **code-inspection story**.
- `Master` has detailed implementation but the competitive table is mostly **reported evidence**.
- `kaggriculture-17` has a large paired local panel, but the headline gain bundles several interventions.
- `More Wheat` has the clearest **model-selection topology** of these four: frozen confirmation worlds, both seats, embedded records and a published rejected challenger.

# 26. Quantitative snapshot

Across the four deeply reviewed submissions:

\[
|C|=194
\]

normalized top-level function ASTs are shared by all four.

Shared-core ratios:

\[
\begin{aligned}
R_{k17}^{core}&=194/209=92.8\%,\\
R_{Master}^{core}&=194/219=88.6\%,\\
R_{Ledger}^{core}&=194/202=96.0\%,\\
R_{MoreWheat}^{core}&=194/213=91.1\%.
\end{aligned}
\]

Branch-unique rates inside this four-agent sample:

\[
0/209=0\%,\quad16/219=7.3\%,\quad6/202=3.0\%,\quad2/213=0.9\%.
\]

For the k17 order search, \(P(7,4)=840\) already exceeds the 800-candidate budget. For its evaluation, observed win-share uplift is +31.8 pp on the curated public panel and +3.7 pp on the recorded ladder sample.

Those three numbers — **shared-core ratio, search coverage, and cross-panel effect size** — summarize the current atlas more meaningfully than a leaderboard snapshot.

# 27. Updated atlas thesis

The biggest update from these three notebooks is **shared-code convergence**.

The dominant pattern in these uploads is:

`mature route/tape chassis`
→ `detect one failure mode`
→ `append a guard/search/repair`
→ `test against closely related public agents`

The disagreement has moved down into local decision layers:

- transaction semantics;
- sale timing;
- exact order sequencing;
- inventory projection;
- seed/fertilizer trimming;
- warehouse closure;
- terminal collection search.

That creates a possible blind spot: a public panel full of the same lineage can reward **lineage-specific best responses**. A genuinely different route generator, explicit opponent-family belief model, robust multi-model planner, or different learned/search substrate becomes more interesting precisely because so much public code now shares the same chassis.

# 28. Protocol for future Deep Dives

Every next solution added to this notebook should answer the same questions:

1. **Policy substrate:** where do candidate actions come from?
2. **Decision locus:** which layer is actually new?
3. **State dependence:** what observations can change the plan online?
4. **Opponent dependence:** observed state, inferred family, predicted actions, or explicit best response?
5. **Planning horizon:** action / turn / phase / season?
6. **Counterfactual mechanism:** threshold, projection, replay, exact simulator, search, learned value?
7. **Lineage:** what is inherited and from whom?
8. **Causal evidence:** is the new mechanism isolated by ablation?
9. **Evaluation topology:** opponents, seed-pairs, seats, holdout, replay RNG preservation?
10. **Failure boundary:** where should this method stop working?
11. **Community implication:** does it deepen an existing branch or open a new one?

The map will be updated from those answers, rather than from notebook titles or leaderboard ranks.

# 29. Sources used for the public-meta snapshot

### Public notebooks / code surfaces

- Kaito Fukami — `40/40 Early Floor | 39/46 Top-10 | v48 Fast Routes`  
  https://www.kaggle.com/code/kaitofukami/40-40-early-floor-39-46-top-10-v48-fast-routes
- Thomas Tschinkel — `93.8% Win Rate Public State Router`  
  https://www.kaggle.com/code/thomastschinkel/kaggriculture-93-8-win-rate-public-state-router
- yamakawanin — `Adaptive Public-State Multi-Route`  
  https://www.kaggle.com/code/yamakawanin/kaggriculture-adaptive-public-state-multi-route
- Igor Zharov — `Multi-Route Farming Agent`  
  https://www.kaggle.com/code/flexonafft/kaggriculture-adaptive-replay-agent
- Ahmed Berat Özer — V23 / V34 / V38 / V43 public lineage  
  https://www.kaggle.com/code/ahmedberatozer/notebook865729c24e  
  https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v34-observed-market-timing  
  https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v38-smarter-feed-stronger-margins  
  https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v43-recovering-lost-harvests
- tetsu2131 — `Market-Smart Farming`  
  https://www.kaggle.com/code/tetsutani/market-smart-farming-kaggriculture
- Yaxon / aurax7 — `Kaggriculture_reactive_router`  
  https://www.kaggle.com/code/aurax7/kaggriculture-reactive-router
- Amey Thakur — `Premium-First Market Agent`  
  https://www.kaggle.com/code/ameythakur20/kaggriculture-premium-first-market-agent
- maverick_ss_26 — `Kaggriculture v1`  
  https://www.kaggle.com/code/maverickss26/kaggriculture-v1
- haodou092 — `Harvest Ledger / V54 Productive Idle Workers`  
  https://www.kaggle.com/code/haodou092/notebookdb6965aa8e

### Public discussions

- RL formulation, opponent ID, evaluation leakage, search-generated targets  
  https://www.kaggle.com/competitions/kaggriculture/discussion/740792
- PPO replay-meta plateau  
  https://www.kaggle.com/competitions/kaggriculture/discussion/736439
- BC/PPO difficulties  
  https://www.kaggle.com/competitions/kaggriculture/discussion/736917
- Replay/tape meta and branching discussion  
  https://www.kaggle.com/competitions/kaggriculture/discussion/738727
- Pattern convergence / RL confirmation  
  https://www.kaggle.com/competitions/kaggriculture/discussion/741792
- Reactive baseline architecture  
  https://www.kaggle.com/competitions/kaggriculture/discussion/739440

### Reviewed source for Deep Dive 001

- Uploaded `kaggriculture-17.ipynb`
- decompressed `main.py` generated by that notebook and statically inspected for the analysis above